# Day 4 — Anthropic (Claude) edition

Two topics today: **tokenization** and **the illusion of memory**.

One honest note on tokenization: `tiktoken` is *OpenAI's* tokenizer and runs locally with no API key. Claude uses a **different** tokenizer that isn't published as a local library, so token counts from `tiktoken` won't exactly match Claude's. We'll use `tiktoken` to *see how tokenization works in general* (the concept is identical across models), then use Anthropic's native **token-counting API** to get true Claude token counts.

## Tokenizing with code (general illustration, via OpenAI's openly-available tokenizer)

In [ ]:
# tiktoken runs locally - no API key needed. Install with: pip install tiktoken
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [ ]:
tokens

In [ ]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

In [ ]:
encoding.decode([326])

### Counting *Claude's* tokens — the Anthropic way

Anthropic gives you a dedicated endpoint to count exactly how many tokens your prompt uses for a given Claude model, without actually running (or paying for) a completion. This is the accurate way to budget tokens on the Anthropic track.

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv(override=True)
client = anthropic.Anthropic()

count = client.messages.count_tokens(
    model="claude-sonnet-4-6",
    messages=[{"role": "user", "content": "Hi my name is Ed and I like banoffee pie"}],
)
print(count.input_tokens, "input tokens for Claude")

# The Illusion of "memory"

For those who haven't seen this — it might be an AHA moment.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('ANTHROPIC_API_KEY')

if not api_key:
    print("No API key was found - add ANTHROPIC_API_KEY=sk-ant-... to your .env file")
elif not api_key.startswith("sk-ant-"):
    print("An API key was found, but it doesn't start sk-ant-; please check your Anthropic key")
else:
    print("API key found and looks good so far!")

### Create the Anthropic client

A lightweight wrapper around HTTP calls to the Claude endpoint.

In [ ]:
import anthropic
client = anthropic.Anthropic()

### A message to Claude is a list of dicts

Remember the Anthropic difference: the **system** prompt is a separate `system=` parameter, not an entry in the messages list. The messages list holds only `user` and `assistant` turns.

In [ ]:
system_prompt = "You are a helpful assistant"

messages = [
    {"role": "user", "content": "Hi! I'm Ed!"}
]

In [ ]:
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    system=system_prompt,
    messages=messages,
)
response.content[0].text

### Now ask a follow-up question — in a brand new call

In [ ]:
messages = [
    {"role": "user", "content": "What's my name?"}
]

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    system=system_prompt,
    messages=messages,
)
response.content[0].text

### Wait, what?? We just told it!

Here's the thing: **every call to an LLM is completely stateless.** It's a brand new call each time, with no memory of prior calls. As AI engineers, it's *our* job to create the impression of memory — by passing the whole conversation back in every time.

In [ ]:
messages = [
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
]

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    system=system_prompt,
    messages=messages,
)
response.content[0].text

## To recap

1. Every call to an LLM is stateless.
2. We pass in the **entire conversation so far** on every call.
3. That creates the illusion the model "remembers" the context.
4. It's a by-product of resending the whole conversation each time.
5. The model just predicts the next tokens; if the sequence contains "My name is Ed" and later "What's my name?", it predicts… Ed.

Chat products work exactly this way — each message resends the full conversation. And yes, you pay for those tokens every time, because you *want* the model to attend over the whole conversation. That's the compute you're paying for.

(Side note for the Anthropic track: this is also why the token-counting API from earlier is useful — as a conversation grows, so does the token cost of every turn.)